In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("covid_toy.csv")
df.head()


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [2]:
df.isnull().sum()


age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["has_covid"])
y = df["has_covid"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

X_train.head()


Train shape: (80, 5)
Test shape: (20, 5)


,age,gender,fever,cough,city
48,66,Male,99.0,Strong,Bangalore
44,20,Male,102.0,Strong,Delhi
58,23,Male,98.0,Strong,Mumbai
15,70,Male,103.0,Strong,Kolkata
51,11,Female,100.0,Strong,Kolkata


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

transformer = ColumnTransformer(
    transformers=[
        ("fever_impute", SimpleImputer(strategy="mean"), ["fever"]),
        ("cough_encode", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
        ("gender_city_ohe", OneHotEncoder(drop="first", handle_unknown="ignore"), ["gender", "city"])
    ],
    remainder="passthrough"
)


In [6]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

print("✅ Train transformed shape:", X_train_transformed.shape)
print("✅ Test transformed shape:", X_test_transformed.shape)


✅ Train transformed shape: (80, 7)
✅ Test transformed shape: (20, 7)


In [7]:
transformer.get_feature_names_out()


array(['fever_impute__fever', 'cough_encode__cough',
       'gender_city_ohe__gender_Male', 'gender_city_ohe__city_Delhi',
       'gender_city_ohe__city_Kolkata', 'gender_city_ohe__city_Mumbai',
       'remainder__age'], dtype=object)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("preprocessing", transformer),
    ("model", LogisticRegression())
])

pipe.fit(X_train, y_train)

print("✅ Pipeline trained successfully!")


✅ Pipeline trained successfully!


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("preprocessing", transformer),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

print("✅ Pipeline trained successfully!")


✅ Pipeline trained successfully!


In [10]:
from sklearn.metrics import accuracy_score

y_pred = pipe.predict(X_test)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))


✅ Accuracy: 0.5


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n✅ Classification Report:")
print(classification_report(y_test, y_pred))


✅ Confusion Matrix:
[[8 3]
 [7 2]]

✅ Classification Report:
              precision    recall  f1-score   support

          No       0.53      0.73      0.62        11
         Yes       0.40      0.22      0.29         9

    accuracy                           0.50        20
   macro avg       0.47      0.47      0.45        20
weighted avg       0.47      0.50      0.47        20

